In [1]:
import re
import pandas as pd
import numpy as np
import networkx as nx
df=pd.read_excel(r'C:\Users\JoeMiller\Python\Supplier_Cleaning_3\IRIS Extract.xlsx')
df["Supplier"]=df["Supplier"].astype(str)

In [3]:
state_mapping = {
    'new south wales': 'nsw', 
    'queensland': 'qld', 
    'victoria': 'vic', 
    'south australia': 'sa', 
    'western australia': 'wa', 
    'tasmania': 'tas', 
    'northern territory': 'nt', 
    'australian capital territory': 'act'
}

ordinal_mapping = {
    'first': '1st',
    'second': '2nd',
    'third': '3rd',
    'fourth': '4th',
    'fifth': '5th',
    'sixth': '6th',
    'seventh': '7th',
    'eighth': '8th',
    'ninth': '9th',
    'tenth': '10th'
}

In [4]:
stop_words = ["and","corporation", "enterprise","incorporated", "us", "international", "llc", "pty", "ltd","limited","australia","australasia"]
stop_words = [word.lower() for word in stop_words]

def normalise_case(supplier_name):
    return supplier_name.lower()
def remove_ampersand(supplier_name):
    return supplier_name.replace('&','and')
def distill_state_abbreviations(supplier_name):
    words=supplier_name.split()
    distilled_words=[state_mapping.get(word, word) for word in words]
    return ' '.join(distilled_words)
def ordinal_map(supplier_name):
    words=supplier_name.split()
    ordinal_words=[ordinal_mapping.get(word, word) for word in words]
    return ' '.join(ordinal_words)
def remove_special_characters(supplier_name):
    return re.sub(r'[^a-zA-Z0-9\s]','',supplier_name)
def remove_my_stopwords(supplier_name):
    words=supplier_name.split()
    return ' '.join([word for word in words if word not in stop_words])
def strip_extra_spaces(supplier_name):
    return re.sub(' +', ' ', supplier_name).strip()

    
def preprocess_supplier_name(supplier_name):
    supplier_name = normalise_case(supplier_name)
    supplier_name = remove_ampersand(supplier_name)
    supplier_name = distill_state_abbreviations(supplier_name)
    supplier_name = ordinal_map(supplier_name)
    supplier_name = remove_special_characters(supplier_name)
    supplier_name = remove_my_stopwords(supplier_name)
    supplier_name = strip_extra_spaces(supplier_name)
    return supplier_name
df["Supplier preprocessed"] = df["Supplier"].apply(preprocess_supplier_name)


no_suppliers=(df['Supplier'].nunique())
no_clean_suppliers=(df['Supplier preprocessed'].nunique())

print(no_suppliers)
print(no_clean_suppliers)

df.to_excel('cleaned_supplier_names IRIS.xlsx', index=False)


7412
6765


In [5]:
from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model.encode(df['Supplier preprocessed'].tolist(), convert_to_tensor=True)

cos_sim_matrix = util.cos_sim(embeddings, embeddings)
threshold = 0.85  # Tune this as needed

supplier_groups = {}
for i, name in enumerate(df['Supplier preprocessed']):
    if name in supplier_groups:
        continue
    similar_idxs = (cos_sim_matrix[i] >= threshold).nonzero()[0]
    group_name = df['Supplier preprocessed'][i]
    for idx in similar_idxs:
        supplier_groups[df['Supplier preprocessed'][idx]] = group_name

df['Supplier grouped'] = df['Supplier preprocessed'].map(supplier_groups)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\JoeMiller\Python\Supplier_cleaning_3\venv\Lib\site-packages\huggingface_hub\file_download.py:144: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\JoeMiller\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

NameError: name 'init_empty_weights' is not defined